# **🏋🏽 Wearable Biomechanics - Temporal Sync**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure Plotly / DataFrame renderer
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("open_dataset_store") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/janithcyapa/OpenDatasetStore.git"])
for pkg in ("ultralytics", "scipy", "cv2"):
    mod = "opencv-python" if pkg == "cv2" else pkg
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", mod])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "cleannjerk_dataset" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import init_store, display_table

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
store = init_store(dataset_path=dataset_path)

### 📊 Dataset Summary & Experiments

In [ ]:
# @title 📊 Dataset Summary
store.summary()
df_experiments = store.list_entries(entry_type="experiments")
display_table(df_experiments)

## Visual Sync Pipeline (Front ↔ Side)

### 🎯 Run Single Visual Sync

In [ ]:
# @title 🎯 Run Single Visual Sync { display-mode: "form" }
from src.sync import visual_estimation_pipeline, save_visual_estimation_results

target_entry_id = "experiments_0001" # @param {type:"string"}
generate_annotated = True # @param {type:"boolean"}
generate_synced = True # @param {type:"boolean"}
plot = True # @param {type:"boolean"}

front_df, side_df, side_lag, front_ann, side_ann, synced_vid = visual_estimation_pipeline(
    entry_id=target_entry_id,
    store=store,
    generate_annotated=generate_annotated,
    generate_synced=generate_synced,
    plot=plot,
)

if front_df is not None or side_df is not None:
    save_visual_estimation_results(
        entry_id=target_entry_id,
        store=store,
        front_df=front_df,
        side_df=side_df,
        side_lag=side_lag,
        front_annotated_path=front_ann,
        side_annotated_path=side_ann,
        synced_video_path=synced_vid,
    )

### 🗑️ Delete Saved Synced/Annotated Videos

In [ ]:
# @title 🗑️ Delete Saved Videos for Experiment
target_entry_id = "experiments_0001" # @param {type:"string"}
store.delete_related_file(entry_type="experiments", entry_id=target_entry_id, label="synced_video")
store.delete_related_file(entry_type="experiments", entry_id=target_entry_id, label="front_annotated_video")
store.delete_related_file(entry_type="experiments", entry_id=target_entry_id, label="side_annotated_video")

### ⚡ Parallel Batch Visual Sync

In [ ]:
# @title ⚡ Parallel Batch Visual Sync
from src.sync import run_batch_visual_estimation

experiments_to_run = [f"experiments_{i:04d}" for i in range(23, 38)]
run_batch_visual_estimation(experiments_to_run, store=store, max_workers=3, plot=False)

## Raw IMU Sync Pipeline (Front Video ↔ 100Hz IMU)

### 🎯 Run Single Front-to-Raw Sync

In [ ]:
# @title 🎯 Run Single Front-to-Raw Sync { display-mode: "form" }
from src.sync import sync_front_to_raw_experiment

target_entry_id = "experiments_0037" # @param {type:"string"}
sync_front_to_raw_experiment(target_entry_id, store=store, plot=True)

### ⚡ Run Batch Front-to-Raw Sync

In [ ]:
# @title ⚡ Run Batch Front-to-Raw Sync { display-mode: "form" }
from src.sync import run_batch_front_to_raw_sync

start_idx = 1 # @param {type:"integer"}
end_idx = 48 # @param {type:"integer"}
run_batch_front_to_raw_sync(start_idx, end_idx, store=store, plot=True)